# 方案四（实验 B）：Qwen3-14B-Instruct LoRA 微调复现

与 `llm-7b.ipynb`（Qwen2.5-7B 全参 SFT）对照，本 Notebook 复现低秩适配（LoRA）实验。

| 实验 | 模型 | 方式 | 关键超参 | 显存 | 测试集 F1 |
| --- | --- | --- | --- | --- | --- |
| A | Qwen2.5-7B-Instruct | 全参 SFT | lr 6e-6、2 epoch、bf16 | 约 50GB（A100 80G） | 0.8042 |
| B | **Qwen3-14B-Instruct** | **LoRA** | rank 64 / alpha 128、lr 1e-4、3 epoch、LoRA 挂所有线性层 | 约 30GB（24G+ 可跑） | **0.5486** |

> 本 Notebook **必须使用 GPU**（建议 32GB 以上；Notebook 会按显卡能力自动选择 bf16/fp16）。脚本产物路径由 `src/config.py` 规定：SFT 数据在 `data/llm/`，权重在 `saves/qwen14b_lora/`，推理结果默认在 `data/llm/Result.csv`。

## Step 0. 环境检查

需要：GPU、`llamafactory[torch,metrics]`、`modelscope`（或设置 `HF_ENDPOINT=https://hf-mirror.com` 从 HuggingFace 下载）。

In [ ]:
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("未找到 nvidia-smi：当前环境没有 NVIDIA GPU，LoRA 训练无法进行。")

import sys
print("Python:", sys.version)

## Step 1. 安装 LLaMA-Factory 与模型下载工具

**国内机器先配清华镜像**可加速 5~10 倍：

```bash
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
pip config set global.trusted-host pypi.tuna.tsinghua.edu.cn
```

然后安装：

```bash
pip install -U "llamafactory[torch,metrics]"
pip install modelscope
llamafactory-cli version
```

In [ ]:
# 首次运行取消注释
# %pip install -q -U "llamafactory[torch,metrics]" modelscope

import importlib.util
print("llamafactory:", "就绪" if importlib.util.find_spec("llamafactory") else "缺失")
print("modelscope:", "就绪" if importlib.util.find_spec("modelscope") else "缺失")
try:
    subprocess.run(["llamafactory-cli", "version"], check=False)
except FileNotFoundError:
    print("llamafactory-cli 未安装，请运行上面的 pip 安装命令")

In [ ]:
# 统一入口：定位项目根目录（notebook/ 的上一级），把 src/ 加入 import 路径
# 脚本产物路径保持原样（config.py 中的 BASE_DIR/outputs, BASE_DIR/data/...）
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 脚本产物写入的根目录（与 src/config.py 保持一致）
ARTIFACT_DIR = PROJECT_ROOT

# 默认使用 0 号 GPU；多卡机器可改成 "1" 等
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

print("项目根目录:", PROJECT_ROOT)
print("产物根目录:", ARTIFACT_DIR)

def run(cmd):
    """用当前 Python 解释器执行命令，工作目录为 PROJECT_ROOT。"""
    print(">>", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, env=os.environ.copy())

In [ ]:
# GPU 环境检查：本实验要求使用 CUDA
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

assert torch.cuda.is_available(), "未检测到 CUDA GPU：请在有 NVIDIA GPU 的实例上运行本 Notebook"
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("显存: %.1f GB" % (props.total_memory / 1e9))
print("CUDA:", torch.version.cuda, "| PyTorch:", torch.__version__)
print("bf16 支持:", torch.cuda.is_bf16_supported())
torch.backends.cudnn.benchmark = True

## Step 2. 构造 SFT 训练数据

把 3229 条评论的标签聚合成 Alpaca 格式（`instruction` / `input` / `output`），并生成 LLaMA-Factory 的 `dataset_info.json`。必须在项目根目录下以 `python -m` 方式运行。

In [ ]:
import pandas as pd

run([sys.executable, "-m", "src.llm.build_sft_dataset"])

import json
sft = json.loads((ARTIFACT_DIR / "data/llm/sft_train.json").read_text(encoding="utf-8"))
print("SFT 样本数:", len(sft))
print("字段:", list(sft[0].keys()))
display(pd.DataFrame(sft).head(2))

## Step 3. 下载 Qwen3-14B-Instruct

通过 ModelScope 缓存到 `/mnt/workspace/models`（云端）或项目下 `models/`（本地）。首次约 28GB。

In [ ]:
from modelscope import snapshot_download

cache_root = ARTIFACT_DIR / "models"
cache_root.mkdir(parents=True, exist_ok=True)
model_dir = snapshot_download("Qwen/Qwen3-14B-Instruct", cache_dir=str(cache_root))
print("模型下载到:", model_dir)
assert Path(model_dir).exists()

## Step 4. 生成本次运行的训练配置

`config/llm_qwen14b_lora.yaml` 中的 `model_name_or_path` 默认写的是 HuggingFace 仓库名。云端通常需要改成 ModelScope 下载后的本地全路径；`dataset_dir` / `output_dir` 也会随当前项目根目录调整。

下面自动生成一份运行时配置 `outputs/llm14b_lora_runtime.yaml`，不修改仓库原配置文件。

In [ ]:
import yaml

base_cfg = yaml.safe_load((PROJECT_ROOT / "config/llm_qwen14b_lora.yaml").read_text(encoding="utf-8"))
base_cfg["model_name_or_path"] = model_dir
base_cfg["dataset_dir"] = str(ARTIFACT_DIR / "data" / "llm")
base_cfg["output_dir"] = str(ARTIFACT_DIR / "saves" / "qwen14b_lora")
base_cfg.setdefault("report_to", "none")

# 按当前 GPU 能力选择 bf16 / fp16（V100、T4 等不支持 bf16）
bf16_ok = torch.cuda.is_bf16_supported()
base_cfg["bf16"] = bf16_ok
base_cfg["fp16"] = not bf16_ok

runtime_cfg = ARTIFACT_DIR / "config" / "llm14b_lora_runtime.yaml"
runtime_cfg.parent.mkdir(parents=True, exist_ok=True)
runtime_cfg.write_text(yaml.safe_dump(base_cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(runtime_cfg.read_text(encoding="utf-8"))

## Step 5. LoRA 训练

- rank 64 / alpha 128 / dropout 0.05，`lora_target: all`；
- batch 4 × 梯度累积 4（等效 16）、lr 1e-4、cosine、3 epoch、gradient checkpointing、bf16；
- 13B 以上模型在 bf16 下约需 30GB 显存；V100/T4 请把 `bf16: true` 改为 `fp16: true`。

```bash
llamafactory-cli train outputs/llm14b_lora_runtime.yaml
```

In [ ]:
run(["llamafactory-cli", "train", str(runtime_cfg)])

In [ ]:
adapter_dir = ARTIFACT_DIR / "saves" / "qwen14b_lora"
print("adapter_config.json:", (adapter_dir / "adapter_config.json").exists())
print("adapter 文件:", [p.name for p in adapter_dir.glob("adapter_*")])
assert (adapter_dir / "adapter_config.json").exists(), "未找到 LoRA 适配器，请检查训练日志"

## Step 6. 推理生成提交文件

LoRA 未合并时，分别指定基础模型与 adapter：

```bash
python src/llm/predict_llm.py --model_path <Qwen3-14B 本地路径> \
    --adapter_path ./saves/qwen14b_lora --backend transformers \
    --max_new_tokens 512 --output data/llm/Result.csv
```

推理侧三个关键细节（代码已处理）：
1. 必须套用 Qwen 聊天模板，否则会出现大量零抽取；
2. 输出解析优先 `ast.literal_eval`，失败则正则抽取 `{...}` 块；
3. 类别/极性过白名单、去空四元组、按 id 去重，空术语写回 `_`。

In [ ]:
result_path = PROJECT_ROOT / "data/llm/Result.csv"

run([sys.executable, "-m", "src.llm.predict_llm",
     "--model_path", str(model_dir),
     "--adapter_path", "saves/qwen14b_lora",
     "--backend", "transformers",
     "--max_new_tokens", "512",
     "--output", str(result_path)])

In [ ]:
import pandas as pd
from pathlib import Path

COLS = ["id", "AspectTerms", "OpinionTerms", "Categories", "Polarities"]
CATS = {"整体", "使用体验", "功效", "价格", "物流", "气味", "包装",
        "真伪", "服务", "其他", "成分", "尺寸", "新鲜度"}
POLS = {"正面", "负面", "中性"}

def validate_result(path):
    p = Path(path)
    assert p.exists(), f"结果文件不存在: {p}"
    df = pd.read_csv(p, header=None, names=COLS, dtype=str)
    df["id"] = df["id"].str.strip().astype(int)
    test_ids = set(pd.read_csv("data/raw/TEST/Test_reviews.csv")["id"].astype(int))
    assert set(df["id"]) == test_ids, "测试集 id 覆盖不完整"
    assert set(df["Categories"].str.strip()) <= CATS, "出现非法类别"
    assert set(df["Polarities"].str.strip()) <= POLS, "出现非法极性"
    for c in COLS[1:]:
        df[c] = df[c].str.strip()
    print(f"[通过] {p}")
    print(f"  行数（四元组数）: {len(df)} | 覆盖 id: {df['id'].nunique()} | 密度: {len(df)/len(test_ids):.2f} 个/条")
    print("  类别分布:", dict(df["Categories"].value_counts()))
    print("  极性分布:", dict(df["Polarities"].value_counts()))
    return df

df = validate_result(ARTIFACT_DIR / "data/llm/Result.csv")
display(df.head(10))

## 结果解读与过拟合对照

- 14B LoRA 3 epoch 的测试 F1 为 **0.5486**，明显低于 7B 全参 2 epoch 的 0.8042；
- 训练 3 epoch 后出现明显**过度抽取**：生成的四元组数量远超训练集密度；
- 小数据（3229 条）上 SFT 轮次宁少勿多。可把 `num_train_epochs` 改为 2 再做一次对照；
- 该模型在最终融合中权重较低（0.5486），但仍提供了与 7B / 管道不同视角的预测。